# 11 — Tools
### LangChain Agents & Tools · Part 1 of 3 (`11_Tools` → `12_Tool_Calling` → `13_Agents`)

Every notebook so far did one thing: text in, text out. A **tool** is how a model gets to *do*
something — look up an order, run a calculation, search the web, hand a case to a human.

## The mental model: a restaurant

| Restaurant | LangChain |
|---|---|
| The **menu** | The tool's **name, description, and input schema** |
| The **customer** (reads the menu, places an order) | The **model** — it only *asks* for a tool, e.g. "`lookup_order` with `order_id="A-1002"`" |
| The **kitchen** (actually cooks it) | **Your Python function** — your code runs it, not the model |

Two things follow, and they shape this whole notebook:

1. **The model never runs your function.** It reads the menu and writes an order. So the *description*
   you write matters as much as the code behind it — a vague menu gets bad orders.
2. **A tool is just four things:** a name, a description, an input schema, and a function.

This notebook is about *building the menu and the kitchen*. The next one (`12_Tool_Calling`) is where
the model starts placing orders.

> 📌 **Worth knowing before you start** (checked against `langchain-core 1.6.7`)
> - **Import from `langchain_core.tools`.** Your earlier script imported `tool` from
>   `langchain_community.tools`. That works — it is the very same object, re-exported — but it routes
>   through the sunset `langchain-community` package for no benefit.
> - **`@tool` does not put your docstring's parameter descriptions into the schema unless you pass
>   `parse_docstring=True`.** It fails silently: the tool works, but the model sees bare parameter
>   names with no explanation. Section 2 shows the difference with real output. (This is a known issue
>   in the LangChain repo, GitHub issue #34292.)
> - **A plain `ValueError` inside a tool is *not* caught by `handle_tool_error`** — only
>   `ToolException` is. Section 6 shows why that matters.
> - **`ShellTool` needs `langchain-experimental`** (also sunset) and has **no safeguards**. Section 10.
> - Most of this notebook needs **no API key and no internet**. Section 10 (web search) needs
>   internet; Section 11 downloads a small local embedding model once.
> - Tools can also receive a hidden `ToolRuntime` parameter to read agent state and context. That is
>   an agent feature, so it is covered in `13_Agents`, not here.

## Setup

In [25]:
# %pip install -q langchain-core pydantic
# %pip install -q langchain-huggingface sentence-transformers          # Section 11 only
# %pip install -q "langchain-community==0.4.2" ddgs "langchain-experimental==0.4.2"   # Section 10 only
%pip install -U ddgs

  Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl (4.0 MB)
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/6.4 MB 6.8 MB/s eta 0:00:01
   ---------------- ----------------------- 2.6/6.4 MB 7.3 MB/s eta 0:00:01
   --------------------------- ------------ 4.5/6.4 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------  6.3/6.4 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 6.4/6.4 MB 8.0 MB/s  0:00:00

   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   -------------------------- ------------- 2/3 [ddgs]
   -------------------------- ------------- 2/3 [ddgs]
   -------------------------- ------------- 2/3 [ddgs]
   -------------------------- ---------

In [1]:
import json, asyncio
from datetime import date, timedelta
from typing import Literal, Type

from pydantic import BaseModel, Field
from langchain_core.tools import tool, StructuredTool, BaseTool, BaseToolkit, ToolException
from langchain_core.messages import ToolMessage

print("Ready — no API key needed for most of this notebook.")


Ready — no API key needed for most of this notebook.


---
## 1. The anatomy of a tool

Start with an ordinary Python function. Wrapping it with `tool(...)` gives it the four parts a model
needs. Notice that LangChain *reads* your function to build them: the name comes from the function
name, the description from the docstring, and the input schema from the type hints.

In [2]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

multiply_tool = tool(multiply)          # same thing as putting @tool above the function

print("name       :", multiply_tool.name)
print("description:", multiply_tool.description)
print("args       :", multiply_tool.args)
print("run it     :", multiply_tool.invoke({"a": 6, "b": 7}))


name       : multiply
description: Multiply two integers and return the result.
args       : {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
run it     : 42


In [3]:
# The full input schema — this is what the model reads to know what arguments it may send.
print(json.dumps(multiply_tool.args_schema.model_json_schema(), indent=2))


{
  "description": "Multiply two integers and return the result.",
  "properties": {
    "a": {
      "title": "A",
      "type": "integer"
    },
    "b": {
      "title": "B",
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "multiply",
  "type": "object"
}


---
## 2. Way one: `@tool` — fastest, for simple functions

You built `multiply` and `add` this way in your first scripts: write the function, add type hints,
add the decorator. Three rules make it work well.

### Rule 1 — A description is mandatory

The model chooses tools by reading descriptions. LangChain refuses to build a tool that has none.

In [4]:
@tool
def get_order_status(order_id: str) -> str:
    """Look up the shipping status of an order by its ID."""
    return f"Order {order_id} is out for delivery."

print(get_order_status.name, "->", get_order_status.description)

# No docstring and no description= -> LangChain refuses:
try:
    @tool
    def no_docs(order_id: str) -> str:
        return order_id
except ValueError as e:
    print("\nValueError:", e)


get_order_status -> Look up the shipping status of an order by its ID.

ValueError: Function must have a docstring if description not provided.


In [5]:
# You can set the name and description explicitly instead of relying on the function name/docstring.
@tool("track_parcel", description="Track a parcel by order ID. Use when the customer asks where their package is.")
def _track(order_id: str) -> str:
    """Internal note for developers (ignored, because description= was given)."""
    return f"Parcel for {order_id}: in transit."

print(_track.name, "|", _track.description)


track_parcel | Track a parcel by order ID. Use when the customer asks where their package is.


### Rule 2 — Use `parse_docstring=True` so parameters get descriptions

Compare what the model receives for the *same* function, with and without it. Look at the `args`.

In [6]:
@tool
def search_kb_default(query: str, top_k: int = 5) -> str:
    """Search the knowledge base.

    Args:
        query: What to search for
        top_k: How many results to return
    """
    return query

@tool(parse_docstring=True)
def search_kb_parsed(query: str, top_k: int = 5) -> str:
    """Search the knowledge base.

    Args:
        query: What to search for
        top_k: How many results to return
    """
    return query

print("DEFAULT (no parse_docstring) — parameters have no descriptions:")
print(json.dumps(search_kb_default.args, indent=2))
print("\nWITH parse_docstring=True — each parameter explained:")
print(json.dumps(search_kb_parsed.args, indent=2))

print("\n--- and the tool DESCRIPTION in each case ---")
print("default:", repr(search_kb_default.description))
print("parsed :", repr(search_kb_parsed.description))


DEFAULT (no parse_docstring) — parameters have no descriptions:
{
  "query": {
    "title": "Query",
    "type": "string"
  },
  "top_k": {
    "default": 5,
    "title": "Top K",
    "type": "integer"
  }
}

WITH parse_docstring=True — each parameter explained:
{
  "query": {
    "description": "What to search for",
    "title": "Query",
    "type": "string"
  },
  "top_k": {
    "default": 5,
    "description": "How many results to return",
    "title": "Top K",
    "type": "integer"
  }
}

--- and the tool DESCRIPTION in each case ---
default: 'Search the knowledge base.\n\nArgs:\n    query: What to search for\n    top_k: How many results to return'
parsed : 'Search the knowledge base.'


Without it, the model sees `top_k: integer` and has to guess what it means. With it, the model is
told. The docstring must follow **Google style** (an `Args:` section) for this to work. Also notice the
default-mode description above swallows the raw `Args:` text into the description; the parsed version
keeps the description clean.

### Rule 3 — Inputs are validated; the return type is not

Arguments are checked against your type hints before your function runs. Reasonable conversions
happen (`"3"` becomes `3`); nonsense is rejected.

In [7]:
print(multiply_tool.invoke({"a": 2, "b": "3"}))      # "3" is converted to the integer 3

try:
    multiply_tool.invoke({"a": 2, "b": "abc"})
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])


6
ValidationError - 1 validation error for multiply


In [8]:
# The OUTPUT is not checked. In one of your first scripts, a function was annotated `-> int`
# but returned an f-string. Nothing complains — so the hint quietly lies.
@tool
def add_with_wrong_hint(a: int, b: int) -> int:
    """Add two numbers."""
    return f"Here is your addition: {a + b}"

print(type(add_with_wrong_hint.invoke({"a": 1, "b": 2})).__name__)   # str, not int


str


Whatever your tool returns is turned into text and handed back to the model. So return something
**clear and self-explanatory** ("Order A-1002 delivered 45 days ago") rather than a bare number, and keep
the return hint honest.

---
## 3. Way two: `StructuredTool` — explicit schema, any existing function

Use it when you want to spell out each input yourself with Pydantic, or when the function already
exists (maybe in a library you don't control) and has no docstring you can edit.

In [9]:
class SubtractInput(BaseModel):
    a: int = Field(description="The number to subtract FROM")
    b: int = Field(description="The number to subtract")

def subtract(a: int, b: int) -> int:
    return a - b          # no docstring needed: the description is supplied below

subtract_tool = StructuredTool.from_function(
    func=subtract,
    name="subtract",
    description="Subtract b from a and return the difference.",
    args_schema=SubtractInput,
)

print(subtract_tool.args)
print(subtract_tool.invoke({"a": 21, "b": 11}))


{'a': {'description': 'The number to subtract FROM', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The number to subtract', 'title': 'B', 'type': 'integer'}}
10


---
## 4. Way three: `BaseTool` — a class, for real logic and state

Subclass it when a tool needs more than a function can hold: its own state, a database connection or
API client, custom error handling, or separate sync and async versions. You implement `_run`.

In [10]:
class DivideInput(BaseModel):
    a: float = Field(description="The numerator")
    b: float = Field(description="The denominator; must not be zero")

class DivideTool(BaseTool):
    name: str = "divide"
    description: str = "Divide a by b and return the quotient."
    args_schema: Type[BaseModel] = DivideInput
    handle_tool_error: bool = True      # explained in Section 6

    def _run(self, a: float, b: float) -> float:
        if b == 0:
            raise ToolException("Cannot divide by zero. Ask the user for a non-zero denominator.")
        return a / b

divide_tool = DivideTool()
print(divide_tool.invoke({"a": 12, "b": 3}))
print(divide_tool.invoke({"a": 12, "b": 0}))      # handled: returns a message instead of crashing


4.0
Cannot divide by zero. Ask the user for a non-zero denominator.


In [11]:
# A class can remember things between calls — something a plain function can't do cleanly.
class GreetInput(BaseModel):
    name: str = Field(description="The person to greet")

class GreetingCounter(BaseTool):
    name: str = "greet"
    description: str = "Greet a person by name and say which greeting number this is."
    args_schema: Type[BaseModel] = GreetInput
    calls: int = 0                                    # state that lives on the tool

    def _run(self, name: str) -> str:
        self.calls += 1
        return f"Hello, {name}! (greeting #{self.calls})"

greeter = GreetingCounter()
print(greeter.invoke({"name": "Asha"}))
print(greeter.invoke({"name": "Ravi"}))


Hello, Asha! (greeting #1)
Hello, Ravi! (greeting #2)


---
## 5. Which way, when?

| | `@tool` | `StructuredTool.from_function` | `BaseTool` subclass |
|---|---|---|---|
| Effort | Lowest | Low | Highest |
| Schema comes from | Type hints + docstring | A Pydantic model you write | A Pydantic model you write |
| Good for | Most tools. Start here | Wrapping an existing function; precise per-field descriptions | State, clients/connections, custom errors, separate sync/async |

**Default to `@tool` with `parse_docstring=True`.** Move to the others only when you hit a reason from
the last row.

---
## 6. When a tool fails — the part tutorials skip

Tools fail constantly: bad IDs, timeouts, missing records. What happens next decides whether your
assistant recovers or crashes. If the failure is turned into a **message**, the model reads it and can
fix its next attempt ("that order doesn't exist — ask the customer to check the number"). If an
**exception escapes**, the whole run dies.

LangChain has one exception type built for this: `ToolException`.

In [12]:
def risky(x: int) -> str:
    """Always fails, to show the three behaviours."""
    raise ToolException("The lookup service is unavailable.")

# 1. Default: the error escapes and would crash an agent run.
t1 = StructuredTool.from_function(risky, name="risky", description="always fails")
try:
    t1.invoke({"x": 1})
except ToolException as e:
    print("1. default           -> raised:", e)

# 2. handle_tool_error=True: the error text is returned to the model as the tool's result.
t2 = StructuredTool.from_function(risky, name="risky", description="always fails", handle_tool_error=True)
print("2. handle=True       ->", t2.invoke({"x": 1}))

# 3. A function: you write the message the model will see.
t3 = StructuredTool.from_function(
    risky, name="risky", description="always fails",
    handle_tool_error=lambda e: f"Tool failed ({e}). Try again later or escalate to a human.",
)
print("3. handle=<function> ->", t3.invoke({"x": 1}))


1. default           -> raised: The lookup service is unavailable.
2. handle=True       -> The lookup service is unavailable.
3. handle=<function> -> Tool failed (The lookup service is unavailable.). Try again later or escalate to a human.


**The gotcha:** `handle_tool_error` only catches `ToolException`. A plain `ValueError`,
`KeyError`, or `TimeoutError` raised inside your tool sails straight through it. Your original `Divide`
tool raised a plain `ValueError` and had no error handling at all, so inside an agent a divide-by-zero
would have ended the whole run. Raise `ToolException` for failures you *expect*, or catch the error
yourself and re-raise it as one.

In [13]:
def lookup_with_valueerror(x: int) -> str:
    """Fails with a plain ValueError."""
    raise ValueError("record not found")

t4 = StructuredTool.from_function(lookup_with_valueerror, name="lookup", description="demo", handle_tool_error=True)
try:
    t4.invoke({"x": 1})
except ValueError as e:
    print("ValueError escaped even though handle_tool_error=True:", e)

# The fix: translate expected failures into ToolException yourself.
def lookup_fixed(x: int) -> str:
    """Fails, but as a ToolException the model can read."""
    try:
        raise ValueError("record not found")
    except ValueError as e:
        raise ToolException(f"Lookup failed: {e}. Check the ID and try again.") from e

t5 = StructuredTool.from_function(lookup_fixed, name="lookup", description="demo", handle_tool_error=True)
print("After the fix ->", t5.invoke({"x": 1}))


ValueError escaped even though handle_tool_error=True: record not found
After the fix -> Lookup failed: record not found. Check the ID and try again.


---
## 7. Sync and async

If your tool waits on the network, an async version lets many calls run at once (you'll feel this in
`13_Agents`). An `async def` tool works with `ainvoke`; give `StructuredTool` both a `func` and a
`coroutine` to support both styles. (Jupyter lets you `await` directly in a cell.)

In [14]:
async def aadd(a: int, b: int) -> int:
    await asyncio.sleep(0.01)          # pretend this is a network call
    return a + b

def add(a: int, b: int) -> int:
    return a + b

add_tool = StructuredTool.from_function(func=add, coroutine=aadd, name="add", description="Add two integers.")

print("sync :", add_tool.invoke({"a": 1, "b": 2}))
print("async:", await add_tool.ainvoke({"a": 1, "b": 2}))


sync : 3
async: 3


In [15]:
# An async-ONLY tool refuses to run synchronously:
@tool
async def async_only(a: int, b: int) -> int:
    """Add two numbers (async only)."""
    return a + b

try:
    async_only.invoke({"a": 1, "b": 2})
except NotImplementedError as e:
    print("NotImplementedError:", e)

print("ainvoke works:", await async_only.ainvoke({"a": 1, "b": 2}))


NotImplementedError: StructuredTool does not support sync invocation.
ainvoke works: 3


---
## 8. What the model actually sees

Since the model only reads the menu, look at the menu. This is the schema (in OpenAI's function
format; LangChain adapts it for other providers when you bind tools in the next notebook). Compare a
careless tool with a careful one — then imagine being the model, choosing between them.

In [16]:
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def do_it(x: str) -> str:
    """Does stuff."""
    return x

@tool(parse_docstring=True)
def lookup_order(order_id: str) -> str:
    """Look up an order's item, status, and delivery date.

    Use this when a customer asks where their order is or what they bought.

    Args:
        order_id: The order ID exactly as the customer wrote it, for example "A-1002".
    """
    return f"(order {order_id})"

print("--- the careless tool ---")
print(json.dumps(convert_to_openai_tool(do_it), indent=2))
print("\n--- the careful tool ---")
print(json.dumps(convert_to_openai_tool(lookup_order), indent=2))


--- the careless tool ---
{
  "type": "function",
  "function": {
    "name": "do_it",
    "description": "Does stuff.",
    "parameters": {
      "properties": {
        "x": {
          "type": "string"
        }
      },
      "required": [
        "x"
      ],
      "type": "object"
    }
  }
}

--- the careful tool ---
{
  "type": "function",
  "function": {
    "name": "lookup_order",
    "description": "Look up an order's item, status, and delivery date. Use this when a customer asks where their order is or what they bought.",
    "parameters": {
      "properties": {
        "order_id": {
          "description": "The order ID exactly as the customer wrote it, for example \"A-1002\".",
          "type": "string"
        }
      },
      "required": [
        "order_id"
      ],
      "type": "object"
    }
  }
}


The careless one says "does stuff" about a parameter called `x`. The careful one says *what it
does, when to use it, and what a valid input looks like*. Writing tool descriptions is prompt
engineering — the model's only information about your tool is these few lines.

**Naming tip:** keep tool names short and simple — letters, digits, underscores (`lookup_order`).
OpenAI's API reference allows only letters, digits, underscores and hyphens (up to 64 characters), so a
name with spaces or a dot can be rejected.

### A preview of what's coming

In the next notebook the model writes a "tool call" — a small dict. When you pass that dict to a tool,
you get back a `ToolMessage`, the standard way a result goes back to the model. Here's one written by
hand:

In [17]:
tool_call = {"name": "lookup_order", "args": {"order_id": "A-1002"}, "id": "call_123", "type": "tool_call"}

result = lookup_order.invoke(tool_call)          # pass the whole call, not just the args
print(type(result).__name__)
print("content     :", result.content)
print("tool_call_id:", result.tool_call_id)       # lets the model match this result to its request


ToolMessage
content     : (order A-1002)
tool_call_id: call_123


---
## 9. Two useful options

**`return_direct=True`** — marks a tool whose output should go straight back to the *user* as the
final answer, instead of back to the model for rewording. (It's just a flag on the tool — an agent loop
is what reads it, so here we only check that it's set.)

**`response_format="content_and_artifact"`** — return two things: short text for the model, and raw
data for *your* code. The model never sees the artifact, so big payloads don't eat its context window.

In [18]:
@tool(return_direct=True)
def get_support_hours() -> str:
    """Return the support team's opening hours."""
    return "Monday to Friday, 9am to 6pm IST."

print("return_direct:", get_support_hours.return_direct)

@tool(response_format="content_and_artifact")
def list_recent_orders(customer_id: str):
    """List a customer's recent order IDs."""
    ids = ["A-1001", "A-1002", "A-1003"]
    return f"Found {len(ids)} recent orders.", ids        # (text for the model, data for your code)

msg = list_recent_orders.invoke({"name": "list_recent_orders", "args": {"customer_id": "C-9"}, "id": "c1", "type": "tool_call"})
print("model sees :", msg.content)
print("your code  :", msg.artifact)


return_direct: True
model sees : Found 3 recent orders.
your code  : ['A-1001', 'A-1002', 'A-1003']


---
## 10. Built-in tools — and why one of them needs a warning label

LangChain ships ready-made tools so you don't write common ones yourself. Two from your original
scripts:

> ⚠️ Both live in `langchain-community`, the sunset package. They still work for learning; for
> production, check the integrations page for a maintained, provider-specific package first.

### Web search — `DuckDuckGoSearchResults`

Needs `pip install -U ddgs` and an internet connection.

In [26]:
try:
    from langchain_community.tools import DuckDuckGoSearchResults

    web_search = DuckDuckGoSearchResults()
    print(web_search.name, "|", web_search.args)
    print(web_search.invoke("latest stable Python release")[:400])
except Exception as e:
    print(f"Skipped ({type(e).__name__}): {str(e)[:200]}")


duckduckgo_results_json | {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}
snippet: Python 3.14.8 is an expedited security release and the eighth maintenance release of 3.14, containing around 354 bugfixes, build improvements and documentation changes from 142 contributors since 3.14.7., title: Python Release Python 3.14.8 | Python.org, link: https://www.python.org/downloads/latest/, snippet: Source and binary executables are signed by the release manager or binary build


### Shell access — `ShellTool` ⚠️

This lets a model run commands on **your machine**. The library's own source says it has *"no
safeguards by default."* Handing an LLM an unrestricted shell means any confusing instruction, or any
malicious text it reads (in a web page, an email, a document), could end up running as a command with
your permissions. Three practical notes:

- It runs through your system shell (`shell=True` in the source). On Windows that is `cmd.exe`, so
  `ls` fails there — use `dir`.
- Pass `ask_human_input=True` and it asks you **y/n before every command** (declining skips the command).
- It needs `langchain-experimental` installed, or it raises an `ImportError`.

In [20]:
try:
    from langchain_community.tools import ShellTool

    shell_tool = ShellTool(ask_human_input=True)      # asks y/n before running anything
    print(shell_tool.name, "|", shell_tool.description)

    # Uncomment to try it — it will ask for confirmation first:
    # print(shell_tool.invoke({"commands": "echo hello"}))
except ImportError as e:
    print("Needs langchain-experimental:", e)


terminal | Run shell commands on this Windows machine.


### The better pattern: narrow tools

Instead of "run any command", offer exactly the few things the model should be able to do, and
constrain the inputs. A `Literal` type turns into an allow-list in the schema, and anything outside it
is rejected before your code runs.

In [21]:
@tool
def check_service_status(service: Literal["api", "database", "payments"]) -> str:
    """Report whether an internal service is operational."""
    status = {"api": "operational", "database": "operational", "payments": "degraded"}
    return f"{service}: {status[service]}"

print(check_service_status.args)                      # note the "enum" allow-list
print(check_service_status.invoke({"service": "payments"}))

try:
    check_service_status.invoke({"service": "coffee-machine"})
except Exception as e:
    print("Rejected before running:", type(e).__name__)


{'service': {'enum': ['api', 'database', 'payments'], 'title': 'Service', 'type': 'string'}}
payments: degraded
Rejected before running: ValidationError


---
## 11. Turning your RAG pipeline into a tool

In `10_RAG_Project`, retrieval ran on *every* message — even "thanks!" triggered a search. As a **tool**,
retrieval becomes something the model *chooses* to do, only when a question needs policy information.
`create_retriever_tool` wraps any retriever as a tool in one call.

In [22]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import create_retriever_tool
from langchain_huggingface import HuggingFaceEmbeddings

policy_store = InMemoryVectorStore(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
policy_store.add_documents([
    Document(page_content="Refunds are issued to the original payment method within 5-7 business days after we receive the returned item.", metadata={"source": "returns_policy.txt"}),
    Document(page_content="Customers may return most items within 30 days of delivery if unused and in original packaging.", metadata={"source": "returns_policy.txt"}),
    Document(page_content="Digital products and gift cards cannot be returned once delivered.", metadata={"source": "returns_policy.txt"}),
    Document(page_content="Standard shipping takes 3-5 business days. Orders over $50 ship free.", metadata={"source": "shipping_policy.txt"}),
])

search_policies = create_retriever_tool(
    policy_store.as_retriever(search_kwargs={"k": 2}),
    name="search_policies",
    description="Search the company's support policies (returns, refunds, shipping). Use for any policy question.",
)

print(search_policies.name, "|", search_policies.args)
print(search_policies.invoke({"query": "how long do refunds take?"}))


d:\Gen Ai\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3083.61it/s]


search_policies | {'query': {'description': 'query to look up in retriever', 'title': 'Query', 'type': 'string'}}
Refunds are issued to the original payment method within 5-7 business days after we receive the returned item.

Customers may return most items within 30 days of delivery if unused and in original packaging.


---
## 12. Toolkits — a group of tools sharing the same resources

A toolkit bundles related tools, and — more usefully — holds what they **share** (a database
connection, an API client). Subclass `BaseToolkit` and implement `get_tools()`. Your earlier
`MathToolkit` was a plain class with a `get_tools()` method; that works, but `BaseToolkit` is the
official base class, and the shared-resources idea is where toolkits earn their keep.

## Real-world capstone: the support desk's toolbox

Four tools for a support assistant: look up an order, check refund eligibility (using the same rules
as the policy documents from `06`–`10`), escalate to a human, and search the policies (Section 11).
The first three share one orders "database" through a toolkit.

In [23]:
TODAY = date.today()

ORDERS = {
    "A-1001": {"item": "Wireless earbuds", "status": "delivered", "delivered_on": TODAY - timedelta(days=10), "digital": False},
    "A-1002": {"item": "Running shoes",    "status": "delivered", "delivered_on": TODAY - timedelta(days=45), "digital": False},
    "A-1003": {"item": "E-book bundle",    "status": "delivered", "delivered_on": TODAY - timedelta(days=2),  "digital": True},
    "A-1004": {"item": "Phone case",       "status": "shipped",   "delivered_on": None,                       "digital": False},
}


class SupportToolkit(BaseToolkit):
    """Order tools that share one orders 'database' and one ticket list."""
    orders: dict
    tickets: list = []

    def get_tools(self) -> list[BaseTool]:
        orders, tickets = self.orders, self.tickets

        def find(order_id: str) -> dict:
            order = orders.get(order_id.strip().upper())
            if order is None:
                raise ToolException(f"No order found with ID '{order_id}'. Ask the customer to double-check the order number.")
            return order

        @tool(parse_docstring=True)
        def lookup_order(order_id: str) -> str:
            """Look up an order's item, status, and delivery date.

            Args:
                order_id: The order ID, for example "A-1001".
            """
            o = find(order_id)
            delivered = o["delivered_on"].isoformat() if o["delivered_on"] else "not yet delivered"
            return f"{order_id.upper()}: {o['item']} | status: {o['status']} | delivered: {delivered}"

        @tool(parse_docstring=True)
        def check_refund_eligibility(order_id: str) -> str:
            """Check whether an order can be refunded under the 30-day return policy.

            Args:
                order_id: The order ID, for example "A-1001".
            """
            o = find(order_id)
            if o["status"] != "delivered":
                return "Not eligible yet: the order has not been delivered."
            if o["digital"]:
                return "Not eligible: digital products cannot be returned once delivered."
            days = (date.today() - o["delivered_on"]).days
            if days > 30:
                return f"Not eligible: delivered {days} days ago, outside the 30-day window."
            return f"Eligible: delivered {days} days ago, inside the 30-day window."

        @tool(parse_docstring=True)
        def escalate_to_human(order_id: str, summary: str) -> str:
            """Hand a case to a human agent when the assistant cannot resolve it.

            Args:
                order_id: The order the case is about.
                summary: One or two sentences describing the problem and what was already tried.
            """
            ticket_id = f"T-{len(tickets) + 1:03d}"
            tickets.append({"ticket": ticket_id, "order_id": order_id, "summary": summary})
            return f"Escalated. Ticket {ticket_id} created; a human will follow up."

        tools = [lookup_order, check_refund_eligibility, escalate_to_human]
        for t in tools:
            t.handle_tool_error = True        # expected failures become messages, not crashes
        return tools


toolkit = SupportToolkit(orders=ORDERS)
toolbox = toolkit.get_tools() + [search_policies]

for t in toolbox:
    print(f"{t.name:26} {list(t.args)}")


lookup_order               ['order_id']
check_refund_eligibility   ['order_id']
escalate_to_human          ['order_id', 'summary']
search_policies            ['query']


### A dry run — playing the model by hand

There's no model in this notebook, so *we* play its part: below is a list of tool calls a model might
plausibly write for "my running shoes arrived ages ago, can I get a refund?" — plus three deliberately
bad ones (an unknown order, a missing argument, a tool that doesn't exist). A small helper runs each
one and turns every outcome, good or bad, into a `ToolMessage`. This is, almost exactly, the loop you'll
build in `12_Tool_Calling`.

In [24]:
def run_tool_calls(tool_calls: list[dict], tools: list[BaseTool]) -> list[ToolMessage]:
    tool_map = {t.name: t for t in tools}
    results = []
    for call in tool_calls:
        if call["name"] not in tool_map:
            results.append(ToolMessage(
                content=f"Error: there is no tool named '{call['name']}'. Available tools: {', '.join(tool_map)}.",
                tool_call_id=call["id"], status="error"))
            continue
        try:
            results.append(tool_map[call["name"]].invoke(call))
        except Exception as e:                       # anything the tool itself didn't handle
            results.append(ToolMessage(content=f"Error ({type(e).__name__}): {e}".splitlines()[0],
                                       tool_call_id=call["id"], status="error"))
    return results


scripted_calls = [
    {"name": "lookup_order",              "args": {"order_id": "A-1002"}, "id": "call_1", "type": "tool_call"},
    {"name": "check_refund_eligibility",  "args": {"order_id": "A-1002"}, "id": "call_2", "type": "tool_call"},
    {"name": "check_refund_eligibility",  "args": {"order_id": "A-1003"}, "id": "call_3", "type": "tool_call"},
    {"name": "search_policies",           "args": {"query": "how long do refunds take?"}, "id": "call_4", "type": "tool_call"},
    {"name": "escalate_to_human",         "args": {"order_id": "A-1002", "summary": "Customer wants a refund outside the 30-day window."}, "id": "call_5", "type": "tool_call"},
    # --- three things that go wrong ---
    {"name": "lookup_order",              "args": {"order_id": "Z-9999"}, "id": "call_6", "type": "tool_call"},   # unknown order
    {"name": "lookup_order",              "args": {},                      "id": "call_7", "type": "tool_call"},   # missing argument
    {"name": "delete_everything",         "args": {},                      "id": "call_8", "type": "tool_call"},   # no such tool
]

for msg in run_tool_calls(scripted_calls, toolbox):
    print(f"[{msg.tool_call_id}] ({msg.status}) {msg.content}")

print("\nTickets created:", toolkit.tickets)


[call_1] (success) A-1002: Running shoes | status: delivered | delivered: 2026-08-25
[call_2] (success) Not eligible: delivered 45 days ago, outside the 30-day window.
[call_3] (success) Not eligible: digital products cannot be returned once delivered.
[call_4] (success) Refunds are issued to the original payment method within 5-7 business days after we receive the returned item.

Customers may return most items within 30 days of delivery if unused and in original packaging.
[call_5] (success) Escalated. Ticket T-001 created; a human will follow up.
[call_6] (error) No order found with ID 'Z-9999'. Ask the customer to double-check the order number.
[call_7] (error) Error (ValidationError): 1 validation error for lookup_order
[call_8] (error) Error: there is no tool named 'delete_everything'. Available tools: lookup_order, check_refund_eligibility, escalate_to_human, search_policies.

Tickets created: [{'ticket': 'T-001', 'order_id': 'A-1002', 'summary': 'Customer wants a refund outside

Read the last three results closely. Each failure produced a **message**, not a crash — and each
message tells the model what to do next ("ask the customer to double-check the order number"). That is
the difference between an assistant that recovers and one that falls over on the first typo.

### Try it yourself
1. Add a fifth tool, `cancel_order`, that only works while the status is `"shipped"`. Make it raise a
   `ToolException` with a helpful message otherwise.
2. Remove `parse_docstring=True` from `escalate_to_human` and compare `convert_to_openai_tool(...)`
   output before and after.
3. Change `check_service_status`'s `Literal` to a plain `str` and call it with `"coffee-machine"`. What
   protects you now?
4. Add `"A-1005"` to `ORDERS` as a delivered, non-digital order from 31 days ago and confirm the
   boundary: is day 30 eligible? Day 31?
5. Write a tool whose description is deliberately vague, and one that's precise, for the same job.
   You'll compare how a real model chooses between them in `12_Tool_Calling`.

---
**Next:** `12_Tool_Calling/` — everything here was the menu and the kitchen. Now the model arrives:
`bind_tools` lets it read the menu, write the tool calls you just wrote by hand, and you'll run the
full loop — call, execute, return the result, get the final answer.
